# Train TinyLlama HelpSteer2 RS-PPO Adapters (ArmoRM)

Trains five independent TinyLlama LoRA specialists with PPO, one per HelpSteer2 attribute, using ArmoRM heads as the reward — the Rewarded-Soups recipe. This notebook is a thin driver: all training logic lives in the already-tested `scripts/train_rs_ppo.py` (RS Table-1 defaults, the verified ArmoRM scorer with its golden-sample and batching checks, Equal-N, plateau detection). We do not reimplement any of that here.

PPO reward and later evaluation both use ArmoRM, so this run is **deliberately circular**: RQ2 is retired; only the upper-bound / geometry / linearity readings are valid.

Scope of this notebook: produce the five adapters and zip them. Computing R is done later in NB05.

## 1. Clone or update the repository

In [ ]:
%cd /content
import os, shutil
repo_path = "/content/master-thesis"
repo_url = "https://github.com/NZhang137/master-thesis.git"
if os.path.isdir(os.path.join(repo_path, ".git")):
    %cd /content/master-thesis
    !git pull
else:
    if os.path.exists(repo_path):
        shutil.rmtree(repo_path)
    !git clone {repo_url} {repo_path}
    %cd /content/master-thesis

## 2. Check the GPU

An A100-40GB is required: the TinyLlama policy and the 4-bit ArmoRM reward model load together during PPO.

In [ ]:
!nvidia-smi

## 3. Install dependencies

Same pinned set as the SFT notebook, plus `trl` for the PPO loop. ArmoRM declares Transformers 4.40.0 and its custom model code relies on that version's internal Llama API, so Transformers, PEFT, and Accelerate stay pinned.

In [ ]:
!pip uninstall -y torchao
!pip install -q -U "pandas==2.2.2" "numpy<2.1" "protobuf>=5.29.1,<6.0.0" "transformers==4.40.0" "peft==0.10.0" "accelerate==0.29.3" "trl==0.8.6" bitsandbytes datasets pyyaml safetensors

Restart the runtime once after installation so Python forgets previously imported Transformers or PyTorch modules, then rerun the repository cell and continue.

## 4. Settings

These are passed to `train_rs_ppo.py` as CLI overrides. Everything else (LoRA rank/alpha/dropout, lr, KL, output length 16–32, ppo_epochs) is fixed at RS Table-1 defaults inside the script and must not be tuned. `total_ppo_steps`, `n_prompts`, `prompt_seed`, and `train_seed` are identical across all five axes (Equal-N) — do not vary them per axis.

In [ ]:
BASE_MODEL   = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
ARMORM_MODEL = "RLHFlow/ArmoRM-Llama3-8B-v0.1"
OUT_DIR      = "results/rs_ppo_armorm_circular/rs_runs"

TOTAL_PPO_STEPS = 200      # short horizon; identical for every axis
BATCH_SIZE      = 64       # RS uses 128; halved for Colab VRAM (use 128 on A100-80GB)
N_PROMPTS       = 2005

print(f"base   = {BASE_MODEL}")
print(f"reward = {ARMORM_MODEL}  (circular: RQ2 retired)")
print(f"out    = {OUT_DIR}")
print(f"steps={TOTAL_PPO_STEPS}  batch={BATCH_SIZE}  n_prompts={N_PROMPTS}")

## 5. Validate the config

Checks axis order and the ArmoRM settings the trainer relies on.

In [ ]:
!python scripts/validate_tinyllama_helpsteer2_config.py

## 6. Create the theta_SFT snapshot

`run_ppo` builds the policy on top of `OUT_DIR/theta_sft/merged` and asserts it exists. This project uses the shortcut theta_SFT = base model (no separate SFT step), so we snapshot TinyLlama-Chat there once. `from_pretrained(...).save_pretrained(...)` is deterministic.

In [ ]:
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

theta_sft = Path(OUT_DIR) / "theta_sft" / "merged"
if (theta_sft / "config.json").exists():
    print(f"theta_SFT already present at {theta_sft}")
else:
    theta_sft.mkdir(parents=True, exist_ok=True)
    m = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float32)
    m.save_pretrained(str(theta_sft))
    AutoTokenizer.from_pretrained(BASE_MODEL).save_pretrained(str(theta_sft))
    del m
    print(f"theta_SFT = base snapshot written to {theta_sft}")

## 7. Manual one-adapter-at-a-time PPO training

Each block trains exactly one axis by calling the tested trainer. ArmoRM (4-bit) and the TinyLlama policy load together, so run **one axis at a time** on a single A100-40GB. The trainer automatically runs the golden-sample head-order check and proves batched == single scoring **before** the first PPO step; if either fails it stops on its own. After each run, check section 9.

The `--circular_armorm_acknowledged` flag is required or the firewall refuses to run. All five axes share the same seeds and step count (Equal-N).

Note: the full head-discriminance gate from the old pipeline (does each ArmoRM head peak on its own axis on labeled text) is **not** in this thin notebook. The two hardest checks — head order via golden sample and batched-vs-single — do run automatically. If you want the discriminance gate as a hard pre-PPO stop, add it before this section.

### Helpfulness

In [ ]:
!python scripts/train_rs_ppo.py \
  --phase ppo \
  --axis helpfulness \
  --reward_model {ARMORM_MODEL} \
  --circular_armorm_acknowledged \
  --out_dir {OUT_DIR} \
  --batch_size {BATCH_SIZE} \
  --total_ppo_steps {TOTAL_PPO_STEPS} \
  --n_prompts {N_PROMPTS}

### Correctness

In [ ]:
!python scripts/train_rs_ppo.py \
  --phase ppo \
  --axis correctness \
  --reward_model {ARMORM_MODEL} \
  --circular_armorm_acknowledged \
  --out_dir {OUT_DIR} \
  --batch_size {BATCH_SIZE} \
  --total_ppo_steps {TOTAL_PPO_STEPS} \
  --n_prompts {N_PROMPTS}

### Coherence

In [ ]:
!python scripts/train_rs_ppo.py \
  --phase ppo \
  --axis coherence \
  --reward_model {ARMORM_MODEL} \
  --circular_armorm_acknowledged \
  --out_dir {OUT_DIR} \
  --batch_size {BATCH_SIZE} \
  --total_ppo_steps {TOTAL_PPO_STEPS} \
  --n_prompts {N_PROMPTS}

### Complexity

In [ ]:
!python scripts/train_rs_ppo.py \
  --phase ppo \
  --axis complexity \
  --reward_model {ARMORM_MODEL} \
  --circular_armorm_acknowledged \
  --out_dir {OUT_DIR} \
  --batch_size {BATCH_SIZE} \
  --total_ppo_steps {TOTAL_PPO_STEPS} \
  --n_prompts {N_PROMPTS}

### Verbosity

In [ ]:
!python scripts/train_rs_ppo.py \
  --phase ppo \
  --axis verbosity \
  --reward_model {ARMORM_MODEL} \
  --circular_armorm_acknowledged \
  --out_dir {OUT_DIR} \
  --batch_size {BATCH_SIZE} \
  --total_ppo_steps {TOTAL_PPO_STEPS} \
  --n_prompts {N_PROMPTS}

## 8. Check the running training

In [ ]:
!pgrep -af "[t]rain_rs_ppo.py" || echo "No PPO training process is running."

Check the ArmoRM download cache if a run is still loading the reward model.

In [ ]:
!du -sh /root/.cache/huggingface/hub/models--RLHFlow--ArmoRM-Llama3-8B-v0.1 2>/dev/null || echo "ArmoRM cache not created yet."

## 9. Inspect reward and plateau per axis

The trainer writes `ppo_log.json` per axis with the full reward log and a plateau interpretation. Two things to confirm for each finished axis:

- the reward curve **moved** and did **not** plateau — `plateau.interpretation` must say "still ascending"; a plateau breaks the short-horizon (upper-bound) premise;
- the **early** `mean_reward` is not far below the ArmoRM usable range (~0.66 raw) — if it is, 16–32-token generations may be too short for ArmoRM to score meaningfully (the open concern from the handoff).

In [ ]:
import json, glob
for path in sorted(glob.glob(f"{OUT_DIR}/ppo_*/ppo_log.json")):
    d = json.load(open(path))
    axis = d["axis"]; log = d["log"]
    early = sum(r["mean_reward"] for r in log[:10]) / max(1, len(log[:10]))
    late  = sum(r["mean_reward"] for r in log[-10:]) / max(1, len(log[-10:]))
    print(f"{axis:12}  early_reward={early:.4f}  late_reward={late:.4f}  "
          f"plateau={d['plateau'].get('reward_plateaued')}")
    print(f"             {d['plateau'].get('interpretation','')}")

## 10. Zip the adapters

Run after all five axes are finished. Contains each `ppo_<axis>/adapter/` plus its log and value head. This is the artifact NB05 consumes to compute R. Keep it out of Git.

In [ ]:
!cd {OUT_DIR} && zip -r /content/rs_ppo_armorm_adapters.zip ppo_*/ -x "*/optimizer*" 
!ls -lh /content/rs_ppo_armorm_adapters.zip
print("Adapters are at:", *[f"{OUT_DIR}/ppo_{a}/adapter" for a in
    ["helpfulness","correctness","coherence","complexity","verbosity"]], sep="\n  ")

## 11. Git safety check

Adapters, snapshots, safetensors, value heads, and zips are generated artifacts. Keep them out of Git.

In [ ]:
!git status